# Chapter 5 — More Is Not Better

## Question

**Does information being present in the context mean the model can use it effectively?**

Falsifiable structure: presence, locatability, usability, and task improvement are separate rungs. This notebook climbs only the first two offline — exact presence and mechanical locatability — and constructs the bundle families that could test the upper rungs. It runs no model, so it proves nothing about behaviour.

## Setup — one critical fact, four bundle conditions

Task fixture: a repair that requires `checkout.timeout_ms = 250`. Conditions follow the chapter: A critical evidence only; B plus irrelevant material; C plus plausible-but-irrelevant material; D plus similar distractors (near-miss values, a superseded doc). Added sizes are kept comparable by construction.

In [ ]:
CRITICAL = 'checkout.timeout_ms = 250'

BASE = ['system: follow repo testing rules.', CRITICAL, 'request: fix the checkout test.']
FILLER_B = ['changelog: release notes for an unrelated widget.', 'history: lunch plans from another channel.']
FILLER_C = ['doc: checkout UI theming guide (plausible, not needed).', 'doc: test-runner installation notes (plausible, not needed).']
FILLER_D = ['config (superseded): checkout.timeout_ms = 500.', 'forum post: try checkout.timeout_ms = 2500.', 'note: checkout.timeout_ms = 25 worked once?']

CONDITIONS = {'A': BASE, 'B': BASE + FILLER_B, 'C': BASE + FILLER_C, 'D': BASE + FILLER_D}
TOKENS = {text: 40 + len(text) for text in BASE + FILLER_B + FILLER_C + FILLER_D}

def render_bundle(texts):
    return '\n'.join(texts)

def mechanical_search(bundle_text, needle):
    """Exact substring search: a structural check, not a model of retrieval."""
    return needle in bundle_text

for name, texts in CONDITIONS.items():
    added = sum(TOKENS[t] for t in texts if t not in BASE)
    print(f"condition {name}: {len(texts)} spans, {sum(TOKENS[t] for t in texts)} fixture tokens (added: {added})")

## Baseline — presence and mechanical locatability in every condition

In [ ]:
print(f"{'cond':>4s} {'present':>7s} {'located':>7s} {'tokens':>6s} {'distractors':>11s} {'near-miss':>9s} {'crit pos':>8s}")
for name, texts in CONDITIONS.items():
    bundle = render_bundle(texts)
    present = CRITICAL in bundle
    located = mechanical_search(bundle, CRITICAL)
    near_miss = sum('timeout_ms' in t for t in texts if t is not CRITICAL and 'timeout_ms' in t and t != CRITICAL)
    distractors = len([t for t in texts if t not in BASE])
    crit_pos = bundle.index(CRITICAL) / len(bundle)
    print(f'{name:>4s} {str(present):>7s} {str(located):>7s} {sum(TOKENS[t] for t in texts):6d} {distractors:11d} {near_miss:9d} {crit_pos:8.2%}')
    assert present and located, 'every condition contains and exposes the critical fact'
print('\nA fact may be present and mechanically locatable while model usability stays an empirical question.')

## Intervention 1 — position fixture (Family B)

The same critical evidence placed at roughly 5, 25, 50, 75, and 95% of rendered length. Actual normalised positions are computed after rendering. No U-curve is reported: the curve is behaviour, and no behaviour was measured.

In [ ]:
FILLER_SPAN = 'filler sentence about unrelated project matters. '

def bundle_with_position(target_share, total_spans=40):
    filler = [FILLER_SPAN] * total_spans
    at = min(total_spans, int(target_share * total_spans))
    spans = filler[:at] + [CRITICAL] + filler[at:]
    text = render_bundle(spans)
    actual = text.index(CRITICAL) / len(text)
    return text, actual

print(f"{'target':>6s} {'actual':>6s} {'length':>7s} {'located':>7s}")
for target in (0.05, 0.25, 0.50, 0.75, 0.95):
    text, actual = bundle_with_position(target)
    print(f'{target:6.0%} {actual:6.1%} {len(text):7d} {str(mechanical_search(text, CRITICAL)):>7s}')
    assert mechanical_search(text, CRITICAL)

## Intervention 2 — classify the bundle, not the behaviour

The notebook may identify distractor-shaped material. It must not claim interference occurred. Language: *potential* interference condition.

In [ ]:
def classify(texts, hard_capacity=20000):
    bundle = render_bundle(texts)
    total = sum(TOKENS[t] for t in texts)
    near_miss = [t for t in texts if 'timeout_ms' in t and t != CRITICAL]
    labels = []
    if total > hard_capacity:
        labels.append('overflow')
    if any(t in FILLER_B + FILLER_C for t in texts):
        labels.append('bloat')
    if near_miss:
        labels.append('potential interference condition (near-miss present; effect untested)')
    return labels

for name, texts in CONDITIONS.items():
    print(f"condition {name}: {classify(texts) or ['no structural flag']}")
assert classify(CONDITIONS['D'])[-1].startswith('potential')
assert classify(CONDITIONS['A']) == []  # A carries no structural flag; behaviour untested anyway

## Absence check — the cheapest diagnostic

Before blaming reasoning, check presence: remove the critical fact and confirm the search goes False.

In [ ]:
no_evidence = render_bundle([t for t in CONDITIONS['B'] if t != CRITICAL])
print('B without critical \u2014 search finds it:', mechanical_search(no_evidence, CRITICAL))
assert not mechanical_search(no_evidence, CRITICAL)
collision = mechanical_search(render_bundle(FILLER_D), CRITICAL)
print("D filler alone triggers substring search ('2500' contains '250'):", collision)
print('Mechanical search is a presence check with sharp edges, not a model of retrieval.')


## Observation — the hypothetical schema, honestly labelled

HYPOTHETICAL OBSERVATIONS — NOT MEASURED MODEL RESULTS. The fields below are the chapter's separated measurements with no values filled in, because filling them in requires the frozen experiment this notebook does not run.

In [ ]:
SCHEMA = ['task_success', 'critical_fact_recovered', 'unsupported_claims', 'latency', 'input_tokens']
print('HYPOTHETICAL OBSERVATIONS \u2014 NOT MEASURED MODEL RESULTS')
for field in SCHEMA:
    print(f'  {field:24s} NOT MEASURED')

## Try it

1. Multiply `FILLER_B` and confirm presence and locatability still hold while tokens grow — volume without a verdict.
2. Swap `FILLER_B` for `FILLER_D` material and watch the classification change while the behavioural claim stays absent.
3. Move `CRITICAL` to the end of condition D and recompute its normalised position.
4. Remove `CRITICAL` entirely and confirm `mechanical_search` returns False — the cheapest diagnostic in the chapter: check presence first.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(mechanical_search(render_bundle(FILLER_D), CRITICAL))  # expect False: absent evidence

## What this demonstrates

- Capacity, presence, locatability, usability, and task improvement are separate questions: every condition here passes the first two and says nothing about the last two.
- Controlled bundle families (A–D plus positioned variants) can be constructed deterministically, with comparable added sizes and computed positions.
- Structural flags (bloat, potential interference condition) describe the bundle's shape, never its behavioural effect.

## What this does not demonstrate

- Context rot in a real model; that D is behaviourally worse than C; a U-shaped position curve.
- Transfer from published benchmark populations to coding agents.
- Any model-specific effective context length.
- That mechanical search predicts model retrieval: substring matching is a presence check, not a usability model.

## Connection to the chapter

Fit is necessary and nowhere near sufficient. If the same information is present but its location changes, then a context bundle cannot be treated as an unordered set:

> If the same information is present but its location changes, then a context bundle cannot be treated as an unordered set.

That is Chapter 6.